In [2]:
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)


DATA_PATH = "IMDB Dataset.csv"

SEED = 42
MAX_SAMPLES = 12000        
MAX_LEN = 128
MAX_VOCAB_SIZE = 20000
MIN_FREQ = 2

BATCH_SIZE = 64
EPOCHS = 11
LR = 2e-4

D_MODEL = 256
NUM_HEADS = 4
D_FF = 512
NUM_LAYERS = 2
DROPOUT = 0.2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


df = pd.read_csv(DATA_PATH)

df = df[["review", "sentiment"]].dropna()

df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

if MAX_SAMPLES and MAX_SAMPLES > 0:
    df = df.sample(n=MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

print(df.head())
print(df["label"].value_counts(normalize=True))


Device: cpu
                                              review sentiment  label
0  I really liked this Summerslam due to the look...  positive      1
1  Not many television shows appeal to quite as m...  positive      1
2  The film quickly gets to a major chase scene w...  negative      0
3  Jane Austen would definitely approve of this o...  positive      1
4  Expectations were somewhat high for me when I ...  negative      0
label
1    0.506583
0    0.493417
Name: proportion, dtype: float64


In [3]:


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)        
    text = re.sub(r"[^a-zA-Z']", " ", text)     
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenize(text):
    return clean_text(text).split()


df["tokens"] = df["review"].apply(simple_tokenize)


train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("Dataset sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
})




Dataset sizes:
{'train': 9600, 'val': 1200, 'test': 1200}


In [4]:

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

counter = Counter()

for tokens in train_df["tokens"]:
    counter.update(tokens)

itos = [PAD_TOKEN, UNK_TOKEN]

for word, freq in counter.most_common(MAX_VOCAB_SIZE - 2):
    if freq >= MIN_FREQ:
        itos.append(word)

stoi = {word: idx for idx, word in enumerate(itos)}

PAD_IDX = stoi[PAD_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]

print("Vocab size:", len(stoi))


def encode_tokens(tokens, max_len=MAX_LEN):
    ids = [stoi.get(token, UNK_IDX) for token in tokens]

    if len(ids) > max_len:
        ids = ids[:max_len]

    attention_mask = [1] * len(ids)

    while len(ids) < max_len:
        ids.append(PAD_IDX)
        attention_mask.append(0)

    return ids, attention_mask


class IMDBDataset(Dataset):
    def __init__(self, dataframe):
        self.tokens = dataframe["tokens"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_tokens(self.tokens[idx])

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset = IMDBDataset(train_df)
val_dataset = IMDBDataset(val_df)
test_dataset = IMDBDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


Vocab size: 20000


In [5]:



class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            # mask: [B, T] -> [B, 1, 1, T]
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, seq_len, self.d_model)

        out = self.out_proj(out)

        return out


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual connection
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)

        # Feed-forward + residual connection
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)

        return x


class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, pad_idx):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_idx
        )

        self.position_embedding = nn.Embedding(
            max_len,
            d_model
        )

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.size()

        positions = torch.arange(
            0,
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(batch_size, seq_len)

        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(positions)

        return token_emb + pos_emb


class EncoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        max_len,
        pad_idx,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = TokenAndPositionEmbedding(
            vocab_size=vocab_size,
            d_model=d_model,
            max_len=max_len,
            pad_idx=pad_idx
        )

        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)

        # masked mean pooling
        mask = attention_mask.unsqueeze(-1)
        x = x * mask

        pooled = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = self.classifier(pooled)

        return logits


model = EncoderOnlyTransformer(
    vocab_size=len(stoi),
    num_classes=2,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX,
    dropout=DROPOUT
).to(DEVICE)

print(model)



criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluation"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1, all_labels, all_preds


best_val_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc, val_f1, _, _ = evaluate(
        model,
        val_loader,
        criterion
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_custom_transformer.pt")
        print("Лучшая модель сохранена")



model.load_state_dict(torch.load("best_custom_transformer.pt", map_location=DEVICE))

test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion
)

print("\nTest results:")
print(f"test_loss = {test_loss:.4f}")
print(f"test_accuracy = {test_acc:.4f}")
print(f"test_f1_macro = {test_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["negative", "positive"],
        digits=4
    )
)


EncoderOnlyTransformer(
  (embedding): TokenAndPositionEmbedding(
    (token_embedding): Embedding(20000, 256, padding_idx=0)
    (position_embedding): Embedding(128, 256)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (layers): ModuleList(
    (0-1): 2 x EncoderLayer(
      (attention): MultiHeadSelfAttention(
        (q_proj): Linear(in_features=256, out_features=256, bias=True)
        (k_proj): Linear(in_features=256, out_features=256, bias=True)
        (v_proj): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (feed_forward): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.2, inplace=False)
          (3): Linear(in_features=512, out_features=256, bias=True)
        )
      )
      (norm1): LayerNorm((256,), eps=1e-0

Training:   0%|          | 0/150 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1/11 | train_loss=0.6456 | train_acc=0.6098 | train_f1=0.6096 | val_loss=0.6347 | val_acc=0.6883 | val_f1=0.6812
Лучшая модель сохранена


Training:   0%|          | 0/150 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:


try:
    del model
    torch.cuda.empty_cache()
except:
    pass


HF_MODEL = "roberta-base"


BATCH_SIZE_ROBERTA = 16
EPOCHS_ROBERTA = 1
LR_ROBERTA = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n=== Fine-tuning предобученной модели: {HF_MODEL} ===")

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL,
    num_labels=2
).to(DEVICE)


class IMDBRoBERTaDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts = dataframe["review"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


train_roberta_dataset = IMDBRoBERTaDataset(train_df, tokenizer, MAX_LEN)
val_roberta_dataset = IMDBRoBERTaDataset(val_df, tokenizer, MAX_LEN)
test_roberta_dataset = IMDBRoBERTaDataset(test_df, tokenizer, MAX_LEN)

train_roberta_loader = DataLoader(
    train_roberta_dataset,
    batch_size=BATCH_SIZE_ROBERTA,
    shuffle=True
)

val_roberta_loader = DataLoader(
    val_roberta_dataset,
    batch_size=BATCH_SIZE_ROBERTA,
    shuffle=False
)

test_roberta_loader = DataLoader(
    test_roberta_dataset,
    batch_size=BATCH_SIZE_ROBERTA,
    shuffle=False
)


optimizer_roberta = torch.optim.AdamW(
    roberta_model.parameters(),
    lr=LR_ROBERTA,
    weight_decay=WEIGHT_DECAY
)

total_training_steps = len(train_roberta_loader) * EPOCHS_ROBERTA
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler_roberta = get_linear_schedule_with_warmup(
    optimizer_roberta,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)


def train_roberta_one_epoch(model, loader, optimizer, scheduler):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc="RoBERTa training"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


def evaluate_roberta(model, loader):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="RoBERTa evaluation"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1, all_labels, all_preds


best_roberta_val_f1 = 0

for epoch in range(1, EPOCHS_ROBERTA + 1):
    train_loss, train_acc, train_f1 = train_roberta_one_epoch(
        roberta_model,
        train_roberta_loader,
        optimizer_roberta,
        scheduler_roberta
    )

    val_loss, val_acc, val_f1, _, _ = evaluate_roberta(
        roberta_model,
        val_roberta_loader
    )

    print(
        f"RoBERTa Epoch {epoch}/{EPOCHS_ROBERTA} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_roberta_val_f1:
        best_roberta_val_f1 = val_f1
        roberta_model.save_pretrained("best_roberta_imdb")
        tokenizer.save_pretrained("best_roberta_imdb")
        print("Лучшая RoBERTa-модель сохранена")


roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "best_roberta_imdb"
).to(DEVICE)

test_loss_roberta, test_acc_roberta, test_f1_roberta, y_true_roberta, y_pred_roberta = evaluate_roberta(
    roberta_model,
    test_roberta_loader
)

print("\nRoBERTa test results:")
print(f"test_loss = {test_loss_roberta:.4f}")
print(f"test_accuracy = {test_acc_roberta:.4f}")
print(f"test_f1_macro = {test_f1_roberta:.4f}")

print("\nRoBERTa classification report:")
print(
    classification_report(
        y_true_roberta,
        y_pred_roberta,
        target_names=["negative", "positive"],
        digits=4
    )
)

# comparison_df = pd.DataFrame([
#     {
#         "model": "Custom encoder-only Transformer",
#         "accuracy": test_acc,
#         "f1_macro": test_f1,
#         "loss": test_loss
#     },
#     {
#         "model": "Fine-tuned RoBERTa",
#         "accuracy": test_acc_roberta,
#         "f1_macro": test_f1_roberta,
#         "loss": test_loss_roberta
#     }
# ])

# print("\n=== Сравнение моделей на test split ===")
# print(comparison_df)


=== Fine-tuning предобученной модели: roberta-base ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa training:   0%|          | 0/600 [00:00<?, ?it/s]

RoBERTa evaluation:   0%|          | 0/75 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Лучшая RoBERTa-модель сохранена


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa evaluation:   0%|          | 0/75 [00:00<?, ?it/s]


RoBERTa test results:
test_loss = 0.2794
test_accuracy = 0.8850
test_f1_macro = 0.8849

RoBERTa classification report:
              precision    recall  f1-score   support

    negative     0.8914    0.8733    0.8823       592
    positive     0.8790    0.8964    0.8876       608

    accuracy                         0.8850      1200
   macro avg     0.8852    0.8848    0.8849      1200
weighted avg     0.8851    0.8850    0.8850      1200

